In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


DISEASE-AWARE ATTENTION

CELL 1 — IMPORT LIBRARIES AND DEFINE MODULE PATHS


This notebook implements the Disease-Aware Attention stage of the
QCDP-BiFormer architecture.

The previous stages produced:

    train_features.pt
        → Trained FiLM-conditioned image features [4491, 768]

    disease_prototypes.pt
        → Disease Prototype Memory [8, 768]

In this module, each image feature will attend to the disease prototype
memory.

Conceptually:

    Image Feature → Query

    Disease Prototypes → Keys + Values

The attention mechanism will produce a disease-aware representation that
combines information from the original image feature and the prototype
memory.

######The output of this module will later be used for bilateral reasoning and cross-eye attention.

---

In [ ]:
# ============================================================
# CELL 1 — IMPORT LIBRARIES AND DEFINE MODULE PATHS
# ============================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

ROOT = "/content/drive/My Drive/Eye Disease/Dataset"

TRAIN_FEATURES_PATH = os.path.join(
    ROOT,
    "train_features.pt"
)

PROTOTYPE_PATH = os.path.join(
    ROOT,
    "disease_prototypes.pt"
)

ATTENTION_SAVE_PATH = os.path.join(
    ROOT,
    "disease_aware_attention.pt"
)

LABEL_COLUMNS = [
    "N", "D", "G", "C",
    "A", "H", "M", "O"
]

NUM_CLASSES = len(LABEL_COLUMNS)
FEATURE_DIM = 768

print("Device:", device)
print("Feature dimension:", FEATURE_DIM)
print("Number of disease prototypes:", NUM_CLASSES)

Device: cuda
Feature dimension: 768
Number of disease prototypes: 8


CELL 2 — LOAD AND VERIFY FEATURE AND PROTOTYPE MEMORY


We now load the learned image features and the Disease Prototype Memory.

The image features represent:

    4,491 training eyes × 768 dimensions

The prototype memory represents:

    8 diseases × 768 dimensions

Before building attention, we verify that both representations share the
same feature dimension.

This is essential because the image features and disease prototypes must
exist in the same learned representation space before attention projections
are applied.

---

In [ ]:
# ============================================================
# CELL 2 — LOAD AND VERIFY FEATURE AND PROTOTYPE MEMORY
# ============================================================

feature_data = torch.load(
    TRAIN_FEATURES_PATH,
    map_location="cpu",
    weights_only=False
)

prototype_memory = torch.load(
    PROTOTYPE_PATH,
    map_location="cpu",
    weights_only=False
)

train_features = feature_data["features"].float()
train_labels = feature_data["labels"].float()

disease_prototypes = (
    prototype_memory["disease_prototypes"]
    .float()
)

assert train_features.shape[1] == FEATURE_DIM
assert disease_prototypes.shape == (
    NUM_CLASSES,
    FEATURE_DIM
)

assert torch.isfinite(train_features).all()
assert torch.isfinite(disease_prototypes).all()

print(
    "Image features:",
    train_features.shape
)

print(
    "Disease prototypes:",
    disease_prototypes.shape
)

print(
    "\nSUCCESS: Feature and prototype memory "
    "are ready for Disease-Aware Attention."
)

Image features: torch.Size([4491, 768])
Disease prototypes: torch.Size([8, 768])

SUCCESS: Feature and prototype memory are ready for Disease-Aware Attention.


CELL 3 — IMPLEMENT DISEASE-AWARE ATTENTION


Disease-Aware Attention allows each image representation to retrieve
relevant information from the Disease Prototype Memory.

Input:

    image_features
        [B, 768]

Prototype memory:

    disease_prototypes
        [8, 768]

The module learns three projections:

    Query:
        image feature → attention query

    Key:
        disease prototype → disease key

    Value:
        disease prototype → disease value

Attention weights are calculated across all 8 disease prototypes.

The resulting disease context is combined with the original image feature
through a residual connection.

Therefore:

    Image Feature
          +
    Disease Prototype Context
          ↓
    Disease-Aware Feature [B, 768]

The prototype memory is initially treated as fixed memory.

The trainable components are the Query, Key, Value and output projections.

---

 CELL 3 — IMPLEMENT NORMALIZED DISEASE-AWARE ATTENTION


This module performs Disease-Aware Attention between each image feature
and the 8 disease prototypes.

The image feature produces a Query representation, while the disease
prototype memory produces Key and Value representations.

To prevent uncontrolled Query-Key magnitude differences from dominating
the attention mechanism, both Query and Key representations are L2
normalized before similarity calculation.

Therefore:

    attention_score = normalized_query × normalized_keyᵀ

This produces cosine similarity scores between each image and the
8 learned disease key representations.

Softmax converts these scores into an attention distribution over the
disease prototypes.

The weighted disease context is projected back to the original feature
dimension and combined with the original image feature using a residual
connection followed by Layer Normalization.

The disease-specific learnable bias used in the previous experiment is
not included in this version because inspection showed that it increased
the existing dominance of the Cataract prototype.

This version preserves the proposed Disease-Aware Attention architecture
while using normalized Query-Key similarity for more stable attention.

---

In [ ]:
# ============================================================
# CELL 3 — IMPLEMENT NORMALIZED DISEASE-AWARE ATTENTION
# ============================================================

class DiseaseAwareAttention(nn.Module):

    def __init__(
        self,
        feature_dim=768,
        attention_dim=256,
        num_diseases=8,
        dropout=0.1
    ):
        super().__init__()

        # ----------------------------------------------------
        # QUERY
        # Image Feature → Query
        # ----------------------------------------------------

        self.query_projection = nn.Linear(
            feature_dim,
            attention_dim
        )

        # ----------------------------------------------------
        # KEY
        # Disease Prototype → Key
        # ----------------------------------------------------

        self.key_projection = nn.Linear(
            feature_dim,
            attention_dim
        )

        # ----------------------------------------------------
        # VALUE
        # Disease Prototype → Value
        # ----------------------------------------------------

        self.value_projection = nn.Linear(
            feature_dim,
            attention_dim
        )

        # ----------------------------------------------------
        # OUTPUT PROJECTION
        # ----------------------------------------------------

        self.output_projection = nn.Linear(
            attention_dim,
            feature_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )

        # ----------------------------------------------------
        # RESIDUAL NORMALIZATION
        # ----------------------------------------------------

        self.norm = nn.LayerNorm(
            feature_dim
        )


    def forward(
        self,
        image_features,
        prototype_memory
    ):

        # ----------------------------------------------------
        # QUERY
        #
        # [B, 768] → [B, 256]
        # ----------------------------------------------------

        queries = self.query_projection(
            image_features
        )


        # ----------------------------------------------------
        # KEYS
        #
        # [8, 768] → [8, 256]
        # ----------------------------------------------------

        keys = self.key_projection(
            prototype_memory
        )


        # ----------------------------------------------------
        # VALUES
        #
        # [8, 768] → [8, 256]
        # ----------------------------------------------------

        values = self.value_projection(
            prototype_memory
        )


        # ----------------------------------------------------
        # L2 NORMALIZE QUERIES AND KEYS
        #
        # This converts the Query-Key comparison into
        # cosine similarity.
        # ----------------------------------------------------

        normalized_queries = F.normalize(
            queries,
            p=2,
            dim=1
        )

        normalized_keys = F.normalize(
            keys,
            p=2,
            dim=1
        )


        # ----------------------------------------------------
        # NORMALIZED QUERY-KEY SIMILARITY
        #
        # [B, 256] @ [256, 8]
        # → [B, 8]
        # ----------------------------------------------------

        attention_scores = (
            normalized_queries
            @ normalized_keys.T
        )


        # ----------------------------------------------------
        # SOFTMAX OVER DISEASE PROTOTYPES
        # ----------------------------------------------------

        attention_weights = F.softmax(
            attention_scores,
            dim=1
        )


        # ----------------------------------------------------
        # WEIGHTED DISEASE CONTEXT
        #
        # [B, 8] @ [8, 256]
        # → [B, 256]
        # ----------------------------------------------------

        disease_context = (
            attention_weights
            @ values
        )


        # ----------------------------------------------------
        # ATTENTION DROPOUT
        # ----------------------------------------------------

        disease_context = self.dropout(
            disease_context
        )


        # ----------------------------------------------------
        # PROJECT BACK TO ORIGINAL FEATURE DIMENSION
        #
        # [B, 256] → [B, 768]
        # ----------------------------------------------------

        disease_context = self.output_projection(
            disease_context
        )


        # ----------------------------------------------------
        # RESIDUAL CONNECTION + LAYER NORMALIZATION
        # ----------------------------------------------------

        disease_aware_features = self.norm(
            image_features
            + disease_context
        )

        attention_similarity = (
            normalized_queries
            @ normalized_keys.T
        )

        return (
          disease_aware_features,
          attention_weights,
          attention_similarity
      )


# ============================================================
# INITIALIZE DISEASE-AWARE ATTENTION
# ============================================================

disease_attention = DiseaseAwareAttention(
    feature_dim=FEATURE_DIM,
    attention_dim=256,
    num_diseases=NUM_CLASSES,
    dropout=0.1
).to(device)


print(disease_attention)

print(
    "\nSUCCESS: Disease-Aware Attention initialized "
    "with normalized Query-Key cosine similarity."
)

DiseaseAwareAttention(
  (query_projection): Linear(in_features=768, out_features=256, bias=True)
  (key_projection): Linear(in_features=768, out_features=256, bias=True)
  (value_projection): Linear(in_features=768, out_features=256, bias=True)
  (output_projection): Linear(in_features=256, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

SUCCESS: Disease-Aware Attention initialized with normalized Query-Key cosine similarity.


 CELL 4 — CONTROLLED FORWARD PASS


We perform the first forward pass through Disease-Aware Attention.

Input:

    FiLM-conditioned image features
    [4491, 768]

Prototype Memory:

    [8, 768]

Expected outputs:

    Disease-Aware Features
    [4491, 768]

    Attention Weights
    [4491, 8]

At this stage, the attention module is not trained yet.

Therefore, this cell only verifies that the mathematical flow of the
Disease-Aware Attention module is structurally correct.

---

In [ ]:
# ============================================================
# CELL 4 — CONTROLLED FORWARD PASS
# ============================================================

disease_attention.eval()

with torch.no_grad():

    sample_features = train_features.to(device)
    prototype_memory_device = (
        disease_prototypes.to(device)
    )

    disease_aware_features, attention_weights = (
        disease_attention(
            sample_features,
            prototype_memory_device
        )
    )

print(
    "Input features:",
    sample_features.shape
)

print(
    "Disease-aware features:",
    disease_aware_features.shape
)

print(
    "Attention weights:",
    attention_weights.shape
)

assert disease_aware_features.shape == (
    train_features.shape[0],
    FEATURE_DIM
)

assert attention_weights.shape == (
    train_features.shape[0],
    NUM_CLASSES
)

print(
    "\nSUCCESS: Forward pass completed correctly."
)

Input features: torch.Size([4491, 768])
Disease-aware features: torch.Size([4491, 768])
Attention weights: torch.Size([4491, 8])

SUCCESS: Forward pass completed correctly.


CELL 5 — ATTENTION INTEGRITY CHECK


Disease-Aware Attention produces one attention distribution across the
8 disease prototypes for every eye.

For each eye:

    attention weights
        [w_N, w_D, w_G, w_C, w_A, w_H, w_M, w_O]

must:

    • Contain only finite values
    • Contain no negative values
    • Sum to approximately 1

These checks verify the Softmax attention mechanism.

---

In [ ]:
# ============================================================
# CELL 5 — ATTENTION INTEGRITY CHECK
# ============================================================

attention_row_sums = (
    attention_weights.sum(dim=1)
)

attention_checks = {
    "Correct attention shape":
        tuple(attention_weights.shape)
        == (len(train_features), NUM_CLASSES),

    "All values finite":
        torch.isfinite(attention_weights).all().item(),

    "No negative attention values":
        torch.all(attention_weights >= 0).item(),

    "All attention rows sum to 1":
        torch.allclose(
            attention_row_sums,
            torch.ones_like(attention_row_sums),
            atol=1e-6
        )
}

for check, result in attention_checks.items():
    print(
        f"{check}: {'PASS' if result else 'FAIL'}"
    )

assert all(attention_checks.values())

print("\nAttention integrity check PASSED.")

Correct attention shape: PASS
All values finite: PASS
No negative attention values: PASS
All attention rows sum to 1: PASS

Attention integrity check PASSED.


 CELL 6 — SAMPLE DISEASE ATTENTION INSPECTION


We inspect the attention distribution for a small number of sample eyes.

Each attention position corresponds to the fixed disease order:

    N, D, G, C, A, H, M, O

Because the Disease-Aware Attention module has not been trained yet,
these values are not interpreted as diagnostic predictions.

This inspection only confirms that the module produces a valid
disease-wise attention distribution for each eye.

---

In [ ]:
# ============================================================
# CELL 6 — SAMPLE DISEASE ATTENTION INSPECTION
# ============================================================

num_samples_to_inspect = 5

sample_attention = (
    attention_weights[
        :num_samples_to_inspect
    ]
    .detach()
    .cpu()
)

sample_labels = (
    train_labels[
        :num_samples_to_inspect
    ]
    .cpu()
)

for i in range(num_samples_to_inspect):

    print("\n" + "=" * 70)
    print(f"Sample Eye {i}")

    true_diseases = [
        LABEL_COLUMNS[j]
        for j in range(NUM_CLASSES)
        if sample_labels[i, j] == 1
    ]

    print(
        "True label(s):",
        true_diseases
    )

    print("\nAttention over prototypes:")

    for disease, weight in zip(
        LABEL_COLUMNS,
        sample_attention[i]
    ):
        print(
            f"{disease}: {weight.item():.4f}"
        )


Sample Eye 0
True label(s): ['N']

Attention over prototypes:
N: 0.1256
D: 0.1268
G: 0.1230
C: 0.1221
A: 0.1262
H: 0.1263
M: 0.1242
O: 0.1258

Sample Eye 1
True label(s): ['N']

Attention over prototypes:
N: 0.1257
D: 0.1271
G: 0.1227
C: 0.1220
A: 0.1261
H: 0.1267
M: 0.1238
O: 0.1259

Sample Eye 2
True label(s): ['D', 'O']

Attention over prototypes:
N: 0.1256
D: 0.1268
G: 0.1229
C: 0.1213
A: 0.1264
H: 0.1269
M: 0.1243
O: 0.1258

Sample Eye 3
True label(s): ['O']

Attention over prototypes:
N: 0.1257
D: 0.1270
G: 0.1230
C: 0.1212
A: 0.1261
H: 0.1265
M: 0.1246
O: 0.1259

Sample Eye 4
True label(s): ['D', 'O']

Attention over prototypes:
N: 0.1252
D: 0.1261
G: 0.1240
C: 0.1228
A: 0.1255
H: 0.1262
M: 0.1248
O: 0.1254


CELL 7 — GRADIENT FLOW VERIFICATION


Before training Disease-Aware Attention, we verify that gradients flow
through all learnable components:

    • Query projection
    • Key projection
    • Value projection
    • Disease-specific bias

This confirms that every trainable component can receive updates during
optimization.

---

In [ ]:
# ============================================================
# CELL 7 — GRADIENT FLOW VERIFICATION
# ============================================================

disease_attention.train()

# Clear any existing gradients
disease_attention.zero_grad()

# Use a small batch for the gradient test
gradient_test_features = (
    train_features[:32]
    .to(device)
)

gradient_test_prototypes = (
    disease_prototypes
    .to(device)
)

gradient_output, gradient_attention = (
    disease_attention(
        gradient_test_features,
        gradient_test_prototypes
    )
)

# Simple scalar objective only for gradient verification
gradient_test_loss = gradient_output.mean()

gradient_test_loss.backward()

gradient_checks = {
    "Query projection gradient":
        disease_attention.query_projection.weight.grad is not None,

    "Key projection gradient":
        disease_attention.key_projection.weight.grad is not None,

    "Value projection gradient":
        disease_attention.value_projection.weight.grad is not None
}

for check, result in gradient_checks.items():
    print(
        f"{check}: {'PASS' if result else 'FAIL'}"
    )

assert all(gradient_checks.values())

print("\nSUCCESS: Gradients flow through all learnable components.")

# Clear temporary gradients
disease_attention.zero_grad()

Query projection gradient: PASS
Key projection gradient: PASS
Value projection gradient: PASS

SUCCESS: Gradients flow through all learnable components.


 CELL 8 — DISEASE-AWARE ATTENTION STRUCTURAL CHECK


The Disease-Aware Attention module is now structurally verified.

We confirm:

    Input:
        [B, 768]

    Disease Prototype Memory:
        [8, 768]

    Query:
        Q(F)

    Keys:
        K(P)

    Values:
        V(P)

    Attention:
        Softmax(QKᵀ / √d_k + B_bias)

    Output:
        F_d = Attention × V(P)
        [B, 768]

The module is now ready to be connected to a trainable disease prediction
objective so that Q, K, V and B_bias can learn disease-relevant attention.

---

In [ ]:
# ============================================================
# CELL 8 — DISEASE-AWARE ATTENTION STRUCTURAL CHECK
# ============================================================

disease_attention.eval()

with torch.no_grad():

    final_test_features = (
        train_features[:64]
        .to(device)
    )

    final_disease_features, final_attention = (
        disease_attention(
            final_test_features,
            disease_prototypes.to(device)
        )
    )

structural_checks = {
    "Input feature dimension":
        final_test_features.shape[1] == FEATURE_DIM,

    "Prototype memory shape":
        tuple(disease_prototypes.shape)
        == (NUM_CLASSES, FEATURE_DIM),

    "Disease-aware output shape":
        tuple(final_disease_features.shape)
        == (64, FEATURE_DIM),

    "Attention output shape":
        tuple(final_attention.shape)
        == (64, NUM_CLASSES),

    "Attention rows sum to 1":
        torch.allclose(
            final_attention.sum(dim=1),
            torch.ones(
                final_attention.size(0),
                device=device
            ),
            atol=1e-6
        ),


}

print("=" * 70)
print("DISEASE-AWARE ATTENTION STRUCTURAL CHECK")
print("=" * 70)

for check, result in structural_checks.items():
    status = "PASS" if result else "FAIL"
    print(f"{status} — {check}")

assert all(structural_checks.values())

print("\n" + "=" * 70)
print("MODULE STRUCTURALLY VERIFIED")
print("=" * 70)

print(
    "\nNext stage: train Disease-Aware Attention "
    "with a multi-label disease prediction objective."
)

DISEASE-AWARE ATTENTION STRUCTURAL CHECK
PASS — Input feature dimension
PASS — Prototype memory shape
PASS — Disease-aware output shape
PASS — Attention output shape
PASS — Attention rows sum to 1

MODULE STRUCTURALLY VERIFIED

Next stage: train Disease-Aware Attention with a multi-label disease prediction objective.


 CELL 9 — PREPARE TRAIN / VALIDATION SPLIT


Disease-Aware Attention must be trained and evaluated on separate data.

The saved feature set contains:

    4491 FiLM-conditioned image features
    [4491, 768]

with corresponding multi-label disease targets:

    [4491, 8]

We create a training and validation split while preserving the alignment
between each feature vector and its multi-label disease target.

The validation set will be used to monitor whether the newly trained
Disease-Aware Attention module generalizes beyond the features used for
optimization.

---

In [ ]:
# ============================================================
# CELL 9 — PREPARE TRAIN / VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

indices = np.arange(len(train_features))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

train_idx = torch.tensor(
    train_idx,
    dtype=torch.long
)

val_idx = torch.tensor(
    val_idx,
    dtype=torch.long
)

daa_train_features = train_features[train_idx]
daa_train_labels = train_labels[train_idx]

daa_val_features = train_features[val_idx]
daa_val_labels = train_labels[val_idx]

print("Total samples:", len(train_features))

print(
    "\nTraining features:",
    daa_train_features.shape
)

print(
    "Validation features:",
    daa_val_features.shape
)

print(
    "\nTraining labels:",
    daa_train_labels.shape
)

print(
    "Validation labels:",
    daa_val_labels.shape
)

assert (
    len(daa_train_features)
    + len(daa_val_features)
    == len(train_features)
)

print("\nSUCCESS: Train/validation split created.")

Total samples: 4491

Training features: torch.Size([3592, 768])
Validation features: torch.Size([899, 768])

Training labels: torch.Size([3592, 8])
Validation labels: torch.Size([899, 8])

SUCCESS: Train/validation split created.


 CELL 10 — CREATE DATASETS AND DATALOADERS


The Disease-Aware Attention module will be trained using mini-batches.

Each batch contains:

    Features:
        [B, 768]

    Multi-label targets:
        [B, 8]

The Disease Prototype Memory remains shared across all batches and will be
provided to the Disease-Aware Attention module during the forward pass.

Only the feature representations and their corresponding disease labels
are stored in the DataLoader.


In [ ]:
# ============================================================
# CELL 10 — CREATE DATASETS AND DATALOADERS
# ============================================================

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

BATCH_SIZE = 64

daa_train_dataset = TensorDataset(
    daa_train_features,
    daa_train_labels
)

daa_val_dataset = TensorDataset(
    daa_val_features,
    daa_val_labels
)

daa_train_loader = DataLoader(
    daa_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

daa_val_loader = DataLoader(
    daa_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

print(
    "Training batches:",
    len(daa_train_loader)
)

print(
    "Validation batches:",
    len(daa_val_loader)
)

sample_batch_features, sample_batch_labels = (
    next(iter(daa_train_loader))
)

print(
    "\nFeature batch shape:",
    sample_batch_features.shape
)

print(
    "Label batch shape:",
    sample_batch_labels.shape
)

Training batches: 57
Validation batches: 15

Feature batch shape: torch.Size([64, 768])
Label batch shape: torch.Size([64, 8])


 CELL 11 — CALCULATE MULTI-LABEL CLASS WEIGHTS


The disease labels are multi-label and potentially imbalanced.

For BCEWithLogitsLoss, we calculate a positive-class weight for each
disease using only the training split.

For each disease:

    pos_weight =
        negative samples / positive samples

This increases the loss contribution of underrepresented positive disease
labels.

The weights are calculated only from the training data to avoid using
validation label statistics during optimization.

---

In [ ]:
# ============================================================
# CELL 11 — CALCULATE MULTI-LABEL CLASS WEIGHTS
# ============================================================

positive_counts = daa_train_labels.sum(dim=0)

negative_counts = (
    len(daa_train_labels)
    - positive_counts
)

pos_weight = (
    negative_counts
    / positive_counts.clamp(min=1)
)

class_weight_df = pd.DataFrame({
    "Disease": LABEL_COLUMNS,
    "Positive Samples":
        positive_counts.numpy().astype(int),
    "Negative Samples":
        negative_counts.numpy().astype(int),
    "pos_weight":
        pos_weight.numpy()
})

display(class_weight_df)

print(
    "\nPositive class weights:",
    pos_weight
)

,Disease,Positive Samples,Negative Samples,pos_weight
0,N,1199,2393,1.995830
1,D,1198,2394,1.998331
2,G,214,3378,15.785047
3,C,232,3360,14.482759
4,A,171,3421,20.005848
5,H,115,3477,30.234783
6,M,183,3409,18.628416
7,O,850,2742,3.225882



Positive class weights: tensor([ 1.9958,  1.9983, 15.7850, 14.4828, 20.0058, 30.2348, 18.6284,  3.2259])


 CELL 12 — DISEASE-AWARE ATTENTION TRAINING MODEL


The Disease-Aware Attention module requires a supervised objective so that
its Query, Key, Value projections and disease-specific bias can learn
disease-relevant attention patterns.

A temporary multi-label classification head is attached to the
Disease-Aware Feature representation.

Training flow:

    FiLM-conditioned Feature
            ↓
    Disease-Aware Attention
            ↓
    Disease-Aware Feature [768]
            ↓
    Multi-Label Classification Head
            ↓
    Disease Logits [8]
            ↓
    BCEWithLogitsLoss

The classification head provides the learning signal required to train the
Disease-Aware Attention module.

The central output of this stage remains the trained Disease-Aware
Attention representation.


---

CELL 12 — DISEASE-AWARE ATTENTION CLASSIFIER


The Disease-Aware Attention module produces a refined feature
representation for each image.

These disease-aware features retain the original feature dimension:

    [B, 768]

A final linear classifier maps each disease-aware feature vector to the
8 multi-label disease logits.

The classifier does not change the attention mechanism itself.

Its purpose is to provide the multi-label disease prediction objective
that trains the Query, Key, Value and output projection parameters of
the Disease-Aware Attention module.

Because Cell 3 has been updated, this classifier is recreated so that it
uses the newly initialized normalized Disease-Aware Attention module.

---

In [ ]:
# ============================================================
# CELL 12 — DISEASE-AWARE ATTENTION CLASSIFIER
# ============================================================

class DiseaseAwareAttentionClassifier(nn.Module):

    def __init__(
        self,
        attention_module,
        feature_dim=768,
        num_classes=8
    ):
        super().__init__()

        self.attention = attention_module

        self.classifier = nn.Linear(
            feature_dim,
            num_classes
        )


    def forward(
        self,
        image_features,
        prototype_memory
    ):

        disease_features, attention_weights = (
            self.attention(
                image_features,
                prototype_memory
            )
        )

        logits = self.classifier(
            disease_features
        )

        return (
            logits,
            disease_features,
            attention_weights
        )


# ============================================================
# INITIALIZE COMPLETE MODEL
# ============================================================

daa_model = DiseaseAwareAttentionClassifier(
    attention_module=disease_attention,
    feature_dim=FEATURE_DIM,
    num_classes=NUM_CLASSES
).to(device)


print(daa_model)

print(
    "\nSUCCESS: Disease-Aware Attention classifier "
    "initialized successfully."
)

DiseaseAwareAttentionClassifier(
  (attention): DiseaseAwareAttention(
    (query_projection): Linear(in_features=768, out_features=256, bias=True)
    (key_projection): Linear(in_features=768, out_features=256, bias=True)
    (value_projection): Linear(in_features=768, out_features=256, bias=True)
    (output_projection): Linear(in_features=256, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (classifier): Linear(in_features=768, out_features=8, bias=True)
)

SUCCESS: Disease-Aware Attention classifier initialized successfully.


 CELL 13 — TRAINING CONFIGURATION


The Disease-Aware Attention training objective is multi-label
classification.

We use:

    BCEWithLogitsLoss

because multiple diseases may be present in the same eye.

Positive class weights are included to account for class imbalance.

The optimizer updates:

    • Query projection
    • Key projection
    • Value projection
    • Disease-specific bias
    • Output projection
    • LayerNorm
    • Temporary classification head

The Disease Prototype Memory remains fixed during this initial attention
training stage.

Training will begin with a controlled sanity check before running the full
training loop.

---

CELL 13 — TRAINING CONFIGURATION


The Disease-Aware Attention model is trained using a multi-label
classification objective.

Because the disease classes are imbalanced, BCEWithLogitsLoss uses
class-specific positive weights.

AdamW is used to optimize all trainable parameters of:

    • Query projection
    • Key projection
    • Value projection
    • Output projection
    • Layer Normalization
    • Final disease classifier

The optimizer is recreated after rebuilding the updated model to ensure
that it references the newly initialized parameters.

---

In [ ]:
# ============================================================
# CELL 13 — TRAINING CONFIGURATION
# ============================================================

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4


# ------------------------------------------------------------
# MULTI-LABEL CLASSIFICATION LOSS
# ------------------------------------------------------------

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight.to(device)
)


# ------------------------------------------------------------
# OPTIMIZER
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    daa_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ------------------------------------------------------------
# PARAMETER COUNT
# ------------------------------------------------------------

trainable_parameters = sum(
    parameter.numel()
    for parameter in daa_model.parameters()
    if parameter.requires_grad
)


print(
    "Loss function:",
    criterion
)

print(
    "\nOptimizer:",
    optimizer.__class__.__name__
)

print(
    "\nLearning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "\nTrainable parameters:",
    f"{trainable_parameters:,}"
)

print(
    "\nSUCCESS: Training configuration is ready."
)

Loss function: BCEWithLogitsLoss()

Optimizer: AdamW

Learning rate: 0.001
Weight decay: 0.0001

Trainable parameters: 795,656

SUCCESS: Training configuration is ready.


In [ ]:
# ============================================================
# REINITIALIZE MODEL FOR KEY DIVERSITY TRAINING
# ============================================================


# ------------------------------------------------------------
# CREATE A FRESH DISEASE-AWARE ATTENTION MODULE
# ------------------------------------------------------------

disease_attention = DiseaseAwareAttention(
    feature_dim=FEATURE_DIM,
    attention_dim=256,
    num_diseases=NUM_CLASSES,
    dropout=0.1
).to(device)


# ------------------------------------------------------------
# CREATE A FRESH CLASSIFIER MODEL
# ------------------------------------------------------------

daa_model = DiseaseAwareAttentionClassifier(
    attention_module=disease_attention,
    feature_dim=FEATURE_DIM,
    num_classes=NUM_CLASSES
).to(device)


# ------------------------------------------------------------
# CREATE A FRESH OPTIMIZER
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    daa_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ------------------------------------------------------------
# VERIFY FRESH INITIALIZATION
# ------------------------------------------------------------

trainable_parameters = sum(
    parameter.numel()
    for parameter in daa_model.parameters()
    if parameter.requires_grad
)


print("=" * 70)
print("FRESH MODEL INITIALIZATION")
print("=" * 70)

print(
    "Disease-Aware Attention:",
    daa_model.attention.__class__.__name__
)

print(
    "Feature dimension:",
    FEATURE_DIM
)

print(
    "Attention dimension:",
    256
)

print(
    "Number of diseases:",
    NUM_CLASSES
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

print(
    "\nSUCCESS: Fresh model and optimizer "
    "are ready for Key Diversity training."
)

FRESH MODEL INITIALIZATION
Disease-Aware Attention: DiseaseAwareAttention
Feature dimension: 768
Attention dimension: 256
Number of diseases: 8
Trainable parameters: 795,656

SUCCESS: Fresh model and optimizer are ready for Key Diversity training.


 CELL 14 — SINGLE-BATCH TRAINING SANITY CHECK


Before running full training, we perform one controlled optimization step.

This verifies that:

    • The model produces valid logits
    • BCEWithLogitsLoss is finite
    • Backpropagation completes successfully
    • The optimizer can update the trainable parameters

This is only a technical sanity check.

The full training process begins in the following cells.


---

In [ ]:
# ============================================================
# CELL 14 — SINGLE-BATCH TRAINING SANITY CHECK
# ============================================================

daa_model.train()

sanity_features, sanity_labels = next(
    iter(daa_train_loader)
)

sanity_features = sanity_features.to(
    device,
    non_blocking=True
)

sanity_labels = sanity_labels.to(
    device,
    non_blocking=True
)

optimizer.zero_grad()

sanity_logits, sanity_features_out, sanity_attention = (
    daa_model(
        sanity_features,
        disease_prototypes.to(device)
    )
)

sanity_loss = criterion(
    sanity_logits,
    sanity_labels
)

assert torch.isfinite(sanity_loss)

sanity_loss.backward()

optimizer.step()

print(
    "Logits shape:",
    sanity_logits.shape
)

print(
    "Disease-aware feature shape:",
    sanity_features_out.shape
)

print(
    "Attention shape:",
    sanity_attention.shape
)

print(
    f"\nSanity-check loss: "
    f"{sanity_loss.item():.6f}"
)

print(
    "\nSUCCESS: One training step completed."
)

Logits shape: torch.Size([64, 8])
Disease-aware feature shape: torch.Size([64, 768])
Attention shape: torch.Size([64, 8])

Sanity-check loss: 1.240526

SUCCESS: One training step completed.


KEY DIVERSITY REGULARIZATION

The diagnostic analysis revealed that the learned disease keys were collapsing into a highly correlated geometric structure. Several disease prototype keys became nearly identical, while the Cataract prototype occupied an opposite direction.

This can cause the query-key similarity mechanism to systematically favor one prototype, resulting in attention collapse.

To encourage the disease prototype keys to remain geometrically distinct, a lightweight Key Diversity Regularization term is introduced during training.

The projected disease keys are first L2-normalized. Their pairwise cosine similarity matrix is then computed. The diagonal elements are ignored because each key is perfectly similar to itself.

The regularization penalizes large pairwise similarities between different disease keys:

Key Diversity Loss = Mean of squared off-diagonal cosine similarities

This auxiliary regularization does not modify the proposed Disease-Aware Attention architecture. It only constrains the training process to encourage the learned disease representations to remain distinct.

The final training objective becomes:

Total Loss = Classification Loss + λ × Key Diversity Loss

where λ controls the influence of the diversity regularization.

DEFINE KEY DIVERSITY REGULARIZATION


The original disease prototypes are highly correlated, and inspection of
the trained attention module showed that the learned Key representations
can develop a pathological geometry.

In particular, several disease keys can become highly similar while one
key becomes strongly separated from the others.

To discourage this collapse, we introduce Key Diversity Regularization.

The projected disease Keys are first L2 normalized.

We then calculate the Key-to-Key cosine similarity matrix:

    K̂ × K̂ᵀ

The diagonal represents self-similarity and is excluded.

The regularization loss penalizes large off-diagonal similarities.

Therefore, the objective encourages different disease keys to maintain
distinct directions in the learned attention space.

This regularization is applied only during training.

It does not alter the proposed Disease-Aware Attention architecture.


In [ ]:
# ============================================================
#  DEFINE KEY DIVERSITY REGULARIZATION
# ============================================================

def key_diversity_loss(
    attention_module,
    prototype_memory
):

    # --------------------------------------------------------
    # PROJECT DISEASE PROTOTYPES INTO KEY SPACE
    #
    # [8, 768] → [8, 256]
    # --------------------------------------------------------

    keys = attention_module.key_projection(
        prototype_memory
    )


    # --------------------------------------------------------
    # L2 NORMALIZE KEYS
    # --------------------------------------------------------

    normalized_keys = F.normalize(
        keys,
        p=2,
        dim=1
    )


    # --------------------------------------------------------
    # KEY-TO-KEY COSINE SIMILARITY MATRIX
    #
    # [8, 256] @ [256, 8]
    # → [8, 8]
    # --------------------------------------------------------

    key_similarity = (
        normalized_keys
        @ normalized_keys.T
    )


    # --------------------------------------------------------
    # REMOVE DIAGONAL SELF-SIMILARITY
    # --------------------------------------------------------

    identity_mask = torch.eye(
        key_similarity.size(0),
        device=key_similarity.device,
        dtype=torch.bool
    )

    off_diagonal_similarity = key_similarity[
        ~identity_mask
    ]


    # --------------------------------------------------------
    # DIVERSITY LOSS
    #
    # Penalize squared similarity between different keys.
    #
    # Lower loss → more diverse key directions.
    # --------------------------------------------------------

    diversity_loss = torch.mean(
        off_diagonal_similarity ** 2
    )


    return diversity_loss


print(
    "Key Diversity Regularization is ready."
)

Key Diversity Regularization is ready.


 VERIFY KEY DIVERSITY REGULARIZATION


Before using Key Diversity Regularization during training, we verify that
the loss can be calculated correctly using the current Disease-Aware
Attention module and disease prototype memory.

The loss must:

    • produce a finite scalar value
    • support gradient computation
    • operate on the projected Key representations
    • not modify the model parameters during inspection

This is only a structural verification.

The actual effect of the regularization will be evaluated after training
by inspecting the learned Key similarity matrix and attention behavior.


In [ ]:
# ============================================================
#  VERIFY KEY DIVERSITY REGULARIZATION
# ============================================================

prototype_memory_device = (
    disease_prototypes
    .to(device)
)


initial_key_diversity_loss = (
    key_diversity_loss(
        attention_module=disease_attention,
        prototype_memory=prototype_memory_device
    )
)


assert initial_key_diversity_loss.ndim == 0

assert torch.isfinite(
    initial_key_diversity_loss
)


print(
    "Initial Key Diversity Loss:",
    initial_key_diversity_loss.item()
)

print(
    "\nSUCCESS: Key Diversity Loss passed "
    "the integrity check."
)

Initial Key Diversity Loss: 0.8865101933479309

SUCCESS: Key Diversity Loss passed the integrity check.


CELL 15 — PARAMETER UPDATE VERIFICATION


A successful backward pass is not enough by itself.

We verify that an optimizer step actually changes the parameters of the
Disease-Aware Attention model.

We compare the Query projection weights before and after one optimization
step.

A non-zero change confirms that training updates are being applied.

---

In [ ]:
# ============================================================
# CELL 15 — PARAMETER UPDATE VERIFICATION
# ============================================================

daa_model.train()

update_features, update_labels = next(
    iter(daa_train_loader)
)

update_features = update_features.to(
    device,
    non_blocking=True
)

update_labels = update_labels.to(
    device,
    non_blocking=True
)

# Save parameter before update
query_weight_before = (
    daa_model
    .attention
    .query_projection
    .weight
    .detach()
    .clone()
)

optimizer.zero_grad()

update_logits, _, _ = daa_model(
    update_features,
    disease_prototypes.to(device)
)

update_loss = criterion(
    update_logits,
    update_labels
)

update_loss.backward()

optimizer.step()

# Save parameter after update
query_weight_after = (
    daa_model
    .attention
    .query_projection
    .weight
    .detach()
    .clone()
)

parameter_change = torch.norm(
    query_weight_after
    - query_weight_before
).item()

print(
    f"Training loss: {update_loss.item():.6f}"
)

print(
    f"Query projection parameter change: "
    f"{parameter_change:.8f}"
)

assert parameter_change > 0

print(
    "\nSUCCESS: Model parameters are updating."
)

Training loss: 1.517557
Query projection parameter change: 0.32333288

SUCCESS: Model parameters are updating.


 CELL 16 — TRAINING AND VALIDATION FUNCTIONS


The training objective combines multi-label classification loss with a
light batch-level attention diversity regularization term.

Classification objective:

    L_classification = BCEWithLogitsLoss

Attention diversity objective:

    For each batch, the attention weights are averaged across samples.

    A_batch = mean(attention, dim=0)

    The entropy of this average prototype distribution is calculated.

    H(A_batch) = -Σ A_batch * log(A_batch)

The diversity term encourages the overall batch to utilize multiple
disease prototypes while still allowing individual samples to have
focused attention.

The total objective is:

    L_total =
        L_classification
        - λ * H(A_batch)

where:

    λ = 0.01

The Disease-Aware Attention architecture itself is unchanged.

During training:

    • Gradients are enabled
    • The total combined loss is backpropagated
    • Model parameters are updated

During validation:

    • Gradients are disabled
    • No parameters are updated
    • The same combined objective is evaluated

---

CELL 16 — TRAINING AND VALIDATION FUNCTIONS


The Disease-Aware Attention model is trained using two objectives.

1. Classification Loss

    BCEWithLogitsLoss measures multi-label disease prediction
    performance.

2. Key Diversity Loss

    This regularization discourages the projected disease Keys from
    collapsing into highly similar directions.

The total objective is:

    Total Loss
        =
    Classification Loss
        +
    λ × Key Diversity Loss

The Key Diversity Loss is calculated directly from the current projected
disease Keys.

Unlike the previous attention entropy experiment, this regularization
targets the learned attention geometry itself rather than directly forcing
the final attention distributions to become artificially uniform.

Training returns:

    • total loss
    • classification loss
    • key diversity loss

This allows classification performance and Key Diversity behavior to be
monitored separately.

---

In [ ]:
# ============================================================
# CELL 16 — TRAINING AND VALIDATION FUNCTIONS
# ============================================================

LAMBDA_KEY_DIVERSITY = 0.01


def train_one_epoch(
    model,
    data_loader,
    criterion,
    optimizer,
    prototype_memory,
    device
):

    model.train()

    running_total_loss = 0.0
    running_classification_loss = 0.0
    running_key_diversity_loss = 0.0

    total_samples = 0


    for batch_features, batch_labels in data_loader:

        batch_features = batch_features.to(
            device,
            non_blocking=True
        )

        batch_labels = batch_labels.to(
            device,
            non_blocking=True
        )


        # ----------------------------------------------------
        # RESET GRADIENTS
        # ----------------------------------------------------

        optimizer.zero_grad()


        # ----------------------------------------------------
        # FORWARD PASS
        # ----------------------------------------------------

        logits, _, _ = model(
            batch_features,
            prototype_memory
        )


        # ----------------------------------------------------
        # CLASSIFICATION LOSS
        # ----------------------------------------------------

        classification_loss = criterion(
            logits,
            batch_labels
        )


        # ----------------------------------------------------
        # KEY DIVERSITY LOSS
        # ----------------------------------------------------

        diversity_loss = key_diversity_loss(
            attention_module=model.attention,
            prototype_memory=prototype_memory
        )


        # ----------------------------------------------------
        # TOTAL OBJECTIVE
        # ----------------------------------------------------

        batch_loss = (
            classification_loss
            + LAMBDA_KEY_DIVERSITY
            * diversity_loss
        )


        # ----------------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------------

        batch_loss.backward()

        optimizer.step()


        # ----------------------------------------------------
        # LOSS ACCUMULATION
        # ----------------------------------------------------

        batch_size = batch_features.size(0)

        running_total_loss += (
            batch_loss.item()
            * batch_size
        )

        running_classification_loss += (
            classification_loss.item()
            * batch_size
        )

        running_key_diversity_loss += (
            diversity_loss.item()
            * batch_size
        )

        total_samples += batch_size


    epoch_total_loss = (
        running_total_loss
        / total_samples
    )

    epoch_classification_loss = (
        running_classification_loss
        / total_samples
    )

    epoch_key_diversity_loss = (
        running_key_diversity_loss
        / total_samples
    )


    return (
        epoch_total_loss,
        epoch_classification_loss,
        epoch_key_diversity_loss
    )



def validate_one_epoch(
    model,
    data_loader,
    criterion,
    prototype_memory,
    device
):

    model.eval()

    running_total_loss = 0.0
    running_classification_loss = 0.0
    running_key_diversity_loss = 0.0

    total_samples = 0


    with torch.no_grad():

        for batch_features, batch_labels in data_loader:

            batch_features = batch_features.to(
                device,
                non_blocking=True
            )

            batch_labels = batch_labels.to(
                device,
                non_blocking=True
            )


            # ------------------------------------------------
            # FORWARD PASS
            # ------------------------------------------------

            logits, _, _ = model(
                batch_features,
                prototype_memory
            )


            # ------------------------------------------------
            # CLASSIFICATION LOSS
            # ------------------------------------------------

            classification_loss = criterion(
                logits,
                batch_labels
            )


            # ------------------------------------------------
            # KEY DIVERSITY LOSS
            # ------------------------------------------------

            diversity_loss = key_diversity_loss(
                attention_module=model.attention,
                prototype_memory=prototype_memory
            )


            # ------------------------------------------------
            # TOTAL VALIDATION OBJECTIVE
            # ------------------------------------------------

            batch_loss = (
                classification_loss
                + LAMBDA_KEY_DIVERSITY
                * diversity_loss
            )


            # ------------------------------------------------
            # LOSS ACCUMULATION
            # ------------------------------------------------

            batch_size = batch_features.size(0)

            running_total_loss += (
                batch_loss.item()
                * batch_size
            )

            running_classification_loss += (
                classification_loss.item()
                * batch_size
            )

            running_key_diversity_loss += (
                diversity_loss.item()
                * batch_size
            )

            total_samples += batch_size


    epoch_total_loss = (
        running_total_loss
        / total_samples
    )

    epoch_classification_loss = (
        running_classification_loss
        / total_samples
    )

    epoch_key_diversity_loss = (
        running_key_diversity_loss
        / total_samples
    )


    return (
        epoch_total_loss,
        epoch_classification_loss,
        epoch_key_diversity_loss
    )


print(
    "Training and validation functions are ready."
)

print(
    f"Key diversity coefficient: "
    f"{LAMBDA_KEY_DIVERSITY}"
)

Training and validation functions are ready.
Key diversity coefficient: 0.01


 CELL 17 — TRAIN DISEASE-AWARE ATTENTION


We now train the Disease-Aware Attention module using the multi-label
classification objective.

The model learns:

    • Q(F): how each image queries disease information
    • K(P): how disease prototypes are represented as attention keys
    • V(P): what disease information is retrieved
    • B_bias: disease-specific attention adjustment
    • Output projection and normalization
    • Temporary multi-label classification mapping

The Disease Prototype Memory itself remains fixed.

Training and validation loss are recorded after every epoch.

The best model is selected using the lowest validation loss.

---

 CELL 17 — TRAIN DISEASE-AWARE ATTENTION


The updated Disease-Aware Attention model is trained from scratch using:

    • normalized Query-Key cosine similarity
    • no disease-specific attention bias
    • multi-label weighted BCE classification loss
    • Key Diversity Regularization

For every epoch, we monitor:

    • total training loss
    • classification training loss
    • key diversity training loss

and the corresponding validation values.

The best model is selected according to validation total loss.

The complete best model state is stored in memory so that it can later
be saved and used to generate the final disease-aware feature memory.

---

In [ ]:
# ============================================================
# CELL 17 — TRAIN DISEASE-AWARE ATTENTION
# ============================================================

NUM_EPOCHS = 20


best_val_loss = float("inf")


training_history = {

    "train_total_loss": [],
    "train_classification_loss": [],
    "train_key_diversity_loss": [],

    "val_total_loss": [],
    "val_classification_loss": [],
    "val_key_diversity_loss": []
}


best_model_state = None


for epoch in range(NUM_EPOCHS):


    # ========================================================
    # TRAIN
    # ========================================================

    (
        train_total_loss,
        train_classification_loss,
        train_key_diversity_loss
    ) = train_one_epoch(

        model=daa_model,

        data_loader=daa_train_loader,

        criterion=criterion,

        optimizer=optimizer,

        prototype_memory=prototype_memory_device,

        device=device
    )


    # ========================================================
    # VALIDATE
    # ========================================================

    (
        val_total_loss,
        val_classification_loss,
        val_key_diversity_loss
    ) = validate_one_epoch(

        model=daa_model,

        data_loader=daa_val_loader,

        criterion=criterion,

        prototype_memory=prototype_memory_device,

        device=device
    )


    # ========================================================
    # SAVE HISTORY
    # ========================================================

    training_history[
        "train_total_loss"
    ].append(
        train_total_loss
    )

    training_history[
        "train_classification_loss"
    ].append(
        train_classification_loss
    )

    training_history[
        "train_key_diversity_loss"
    ].append(
        train_key_diversity_loss
    )


    training_history[
        "val_total_loss"
    ].append(
        val_total_loss
    )

    training_history[
        "val_classification_loss"
    ].append(
        val_classification_loss
    )

    training_history[
        "val_key_diversity_loss"
    ].append(
        val_key_diversity_loss
    )


    # ========================================================
    # DISPLAY RESULTS
    # ========================================================

    print(
        f"Epoch [{epoch + 1:02d}/{NUM_EPOCHS}]"
    )

    print(
        f"Train Total: {train_total_loss:.6f} | "
        f"Cls: {train_classification_loss:.6f} | "
        f"KeyDiv: {train_key_diversity_loss:.6f}"
    )

    print(
        f"Val Total: {val_total_loss:.6f} | "
        f"Cls: {val_classification_loss:.6f} | "
        f"KeyDiv: {val_key_diversity_loss:.6f}"
    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_total_loss < best_val_loss:

        best_val_loss = val_total_loss


        best_model_state = {

            name: parameter.detach()
            .cpu()
            .clone()

            for name, parameter in
            daa_model.state_dict().items()
        }


        print(
            "✓ Best validation model updated"
        )


print("\n" + "=" * 70)

print(
    "TRAINING COMPLETE"
)

print("=" * 70)


print(
    f"Best Validation Total Loss: "
    f"{best_val_loss:.6f}"
)

Epoch [01/20]
Train Total: 1.114629 | Cls: 1.108843 | KeyDiv: 0.578588
Val Total: 1.029129 | Cls: 1.024088 | KeyDiv: 0.504097
✓ Best validation model updated
Epoch [02/20]
Train Total: 0.966861 | Cls: 0.962115 | KeyDiv: 0.474619
Val Total: 0.935945 | Cls: 0.931851 | KeyDiv: 0.409423
✓ Best validation model updated
Epoch [03/20]
Train Total: 0.893280 | Cls: 0.889745 | KeyDiv: 0.353539
Val Total: 0.902879 | Cls: 0.900334 | KeyDiv: 0.254509
✓ Best validation model updated
Epoch [04/20]
Train Total: 0.862680 | Cls: 0.859730 | KeyDiv: 0.294980
Val Total: 0.873608 | Cls: 0.870739 | KeyDiv: 0.286889
✓ Best validation model updated
Epoch [05/20]
Train Total: 0.849085 | Cls: 0.845940 | KeyDiv: 0.314498
Val Total: 0.872432 | Cls: 0.869216 | KeyDiv: 0.321510
✓ Best validation model updated
Epoch [06/20]
Train Total: 0.846106 | Cls: 0.843141 | KeyDiv: 0.296485
Val Total: 1.065913 | Cls: 1.062483 | KeyDiv: 0.342936
Epoch [07/20]
Train Total: 0.884132 | Cls: 0.880452 | KeyDiv: 0.368053
Val Total: 0.

CELL 18 — SAVE BEST DISEASE-AWARE ATTENTION MODEL


After training, we save the best model based on validation loss.

The checkpoint contains:

    • Complete trained model state
    • Disease-Aware Attention parameters
    • Temporary classifier parameters
    • Best validation loss
    • Training history
    • Disease label ordering
    • Feature and prototype dimensions

The Disease Prototype Memory is not duplicated inside this checkpoint.
It remains stored separately as:

    disease_prototypes.pt

Together, the prototype memory and the trained Disease-Aware Attention
checkpoint provide the learned components required for the next stages
of the QCDP-BiFormer pipeline.

---

In [ ]:
# ============================================================
# CELL 18 — SAVE BEST DISEASE-AWARE ATTENTION MODEL
# ============================================================

BEST_DAA_MODEL_PATH = os.path.join(
    ROOT,
    "disease_aware_attention_best.pt"
)

checkpoint = {
    "model_state_dict": best_model_state,

    "best_val_loss": best_val_loss,

    "training_history": training_history,

    "label_columns": LABEL_COLUMNS,

    "feature_dim": FEATURE_DIM,

    "attention_dim": 256,

    "num_diseases": NUM_CLASSES,

    "prototype_shape":
        tuple(disease_prototypes.shape),

    "architecture": {
        "query": "768 -> 256",
        "key": "768 -> 256",
        "value": "768 -> 256",
        "output": "256 -> 768",
        "disease_bias": True,
        "residual_connection": True,
        "layer_norm": True
    }
}

torch.save(
    checkpoint,
    BEST_DAA_MODEL_PATH
)

assert os.path.exists(
    BEST_DAA_MODEL_PATH
)

print(
    "Best Disease-Aware Attention model saved."
)

print(
    "Path:",
    BEST_DAA_MODEL_PATH
)

print(
    "Best Validation Loss:",
    best_val_loss
)

Best Disease-Aware Attention model saved.
Path: /content/drive/My Drive/Eye Disease/Dataset/disease_aware_attention_best.pt
Best Validation Loss: 0.8359162975206258


 CELL 19 — RELOAD BEST DISEASE-AWARE ATTENTION MODEL


During training, the best model was selected using the lowest validation
loss.

Best result:

    Epoch 18
    Best Validation Loss: 0.823291

We now reload this saved checkpoint so that all subsequent feature
generation uses the best learned Disease-Aware Attention parameters,
rather than the parameters remaining after the final training epoch.

The checkpoint contains:

    • Trained Disease-Aware Attention parameters
    • Temporary classifier parameters
    • Disease-specific learnable bias
    • Training history
    • Architecture metadata

The trained model will be used to generate the final Disease-Aware
Feature representations for the complete feature set.

---

In [ ]:
# ============================================================
# CELL 19 — RELOAD BEST DISEASE-AWARE ATTENTION MODEL
# ============================================================

checkpoint = torch.load(
    BEST_DAA_MODEL_PATH,
    map_location=device,
    weights_only=False
)

print("Checkpoint loaded successfully.")

print(
    "\nBest validation loss:",
    checkpoint["best_val_loss"]
)

print(
    "Feature dimension:",
    checkpoint["feature_dim"]
)

print(
    "Attention dimension:",
    checkpoint["attention_dim"]
)

print(
    "Number of diseases:",
    checkpoint["num_diseases"]
)

print(
    "\nLabel order:",
    checkpoint["label_columns"]
)

Checkpoint loaded successfully.

Best validation loss: 0.8359162975206258
Feature dimension: 768
Attention dimension: 256
Number of diseases: 8

Label order: ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']


 CELL 20 — RESTORE AND VERIFY BEST MODEL


We recreate the Disease-Aware Attention training model and load the
saved best parameters.

After restoration, we verify:

    • All expected parameters are present
    • The model loads without missing or unexpected keys
    • The model is placed in evaluation mode

The temporary classifier remains attached only because it is part of the
saved training checkpoint.

For the next pipeline stage, our main interest is the trained:

    Disease-Aware Attention module

and its resulting Disease-Aware Feature representation.

---

In [ ]:
# ============================================================
# CELL 20 — RESTORE AND VERIFY BEST MODEL
# ============================================================

# Recreate the model architecture
best_daa_model = DiseaseAwareAttentionClassifier(
    attention_module=DiseaseAwareAttention(
        feature_dim=FEATURE_DIM,
        attention_dim=256,
        num_diseases=NUM_CLASSES,
        dropout=0.1
    ),
    feature_dim=FEATURE_DIM,
    num_classes=NUM_CLASSES
).to(device)

# Load best trained parameters
load_result = best_daa_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

best_daa_model.eval()

print(
    "Missing keys:",
    load_result.missing_keys
)

print(
    "Unexpected keys:",
    load_result.unexpected_keys
)

assert len(load_result.missing_keys) == 0
assert len(load_result.unexpected_keys) == 0

print("\nSUCCESS: Best model restored correctly.")

Missing keys: []
Unexpected keys: []

SUCCESS: Best model restored correctly.


 CELL 21 — GENERATE FINAL DISEASE-AWARE FEATURES


We now pass all 4491 FiLM-conditioned image features through the
best trained Disease-Aware Attention module.

Input:

    FiLM-conditioned features
    [4491, 768]

Disease Prototype Memory:

    [8, 768]

Output:

    Disease-Aware Features
    [4491, 768]

    Disease Attention Weights
    [4491, 8]

These representations are generated using the best validation model and
will serve as the learned output of the Disease-Aware Attention stage.

The feature order is preserved exactly so that each generated feature
remains aligned with its original sample and multi-label target.

---

In [ ]:
# ============================================================
# CELL 21 — GENERATE FINAL DISEASE-AWARE FEATURES
# ============================================================

best_attention_module = best_daa_model.attention

best_attention_module.eval()

all_disease_aware_features = []
all_trained_attention_weights = []

feature_generation_loader = DataLoader(
    train_features,
    batch_size=128,
    shuffle=False,
    pin_memory=True
)

prototype_memory_device = (
    disease_prototypes
    .to(device)
)

with torch.no_grad():

    for batch_features in feature_generation_loader:

        batch_features = batch_features.to(
            device,
            non_blocking=True
        )

        batch_disease_features, batch_attention = (
            best_attention_module(
                batch_features,
                prototype_memory_device
            )
        )

        all_disease_aware_features.append(
            batch_disease_features.cpu()
        )

        all_trained_attention_weights.append(
            batch_attention.cpu()
        )


all_disease_aware_features = torch.cat(
    all_disease_aware_features,
    dim=0
)

all_trained_attention_weights = torch.cat(
    all_trained_attention_weights,
    dim=0
)

print(
    "Final disease-aware features:",
    all_disease_aware_features.shape
)

print(
    "Final attention weights:",
    all_trained_attention_weights.shape
)

assert tuple(all_disease_aware_features.shape) == (
    len(train_features),
    FEATURE_DIM
)

assert tuple(all_trained_attention_weights.shape) == (
    len(train_features),
    NUM_CLASSES
)

print(
    "\nSUCCESS: Final Disease-Aware features generated."
)

Final disease-aware features: torch.Size([4491, 768])
Final attention weights: torch.Size([4491, 8])

SUCCESS: Final Disease-Aware features generated.


 CELL 22 — INSPECT TRAINED DISEASE ATTENTION


Before training, Disease-Aware Attention produced an almost uniform
distribution across the 8 disease prototypes:

    approximately 0.125 per prototype

After supervised training, we inspect the learned attention distributions.

For each sample, attention values still represent prototype relevance and
must not be interpreted directly as final disease probabilities.

The purpose of this inspection is to verify that attention has become
input-dependent and is no longer uniformly distributed across all
prototypes.

---

In [ ]:
# ============================================================
# CELL 22 — INSPECT TRAINED DISEASE ATTENTION
# ============================================================

num_samples_to_inspect = 5

for i in range(num_samples_to_inspect):

    print("\n" + "=" * 70)
    print(f"Sample Eye {i}")

    true_diseases = [
        LABEL_COLUMNS[j]
        for j in range(NUM_CLASSES)
        if train_labels[i, j] == 1
    ]

    print(
        "True label(s):",
        true_diseases
    )

    print("\nTrained attention over prototypes:")

    for disease, weight in zip(
        LABEL_COLUMNS,
        all_trained_attention_weights[i]
    ):
        print(
            f"{disease}: {weight.item():.4f}"
        )

    print(
        "\nMaximum attended prototype:",
        LABEL_COLUMNS[
            torch.argmax(
                all_trained_attention_weights[i]
            ).item()
        ]
    )


Sample Eye 0
True label(s): ['N']

Trained attention over prototypes:
N: 0.0653
D: 0.0603
G: 0.1326
C: 0.4116
A: 0.0723
H: 0.0619
M: 0.1359
O: 0.0599

Maximum attended prototype: C

Sample Eye 1
True label(s): ['N']

Trained attention over prototypes:
N: 0.0648
D: 0.0596
G: 0.1347
C: 0.4086
A: 0.0730
H: 0.0611
M: 0.1387
O: 0.0596

Maximum attended prototype: C

Sample Eye 2
True label(s): ['D', 'O']

Trained attention over prototypes:
N: 0.0653
D: 0.0598
G: 0.1408
C: 0.3974
A: 0.0737
H: 0.0617
M: 0.1416
O: 0.0598

Maximum attended prototype: C

Sample Eye 3
True label(s): ['O']

Trained attention over prototypes:
N: 0.0643
D: 0.0591
G: 0.1377
C: 0.4046
A: 0.0733
H: 0.0607
M: 0.1411
O: 0.0593

Maximum attended prototype: C

Sample Eye 4
True label(s): ['D', 'O']

Trained attention over prototypes:
N: 0.0641
D: 0.0582
G: 0.1418
C: 0.3971
A: 0.0747
H: 0.0594
M: 0.1456
O: 0.0590

Maximum attended prototype: C


 CELL 23 — SAVE DISEASE-AWARE FEATURE MEMORY


The trained Disease-Aware Attention module has now transformed the
FiLM-conditioned feature memory into Disease-Aware representations.

We perform a final integrity check to verify:

    • Correct number of samples
    • Correct feature dimension
    • Correct attention dimension
    • All values are finite
    • Every attention distribution sums to approximately 1

The resulting memory is saved for the next module.

Saved artifacts:

    disease_aware_features.pt
    disease_attention_weights.pt

The feature order is preserved from train_features.pt so that sample
alignment with labels and paired-eye information remains unchanged.

The next stage can therefore begin from:

    disease_aware_features.pt
                +
    disease_attention_weights.pt
                +
    paired-eye information
                ↓
    Cross-Eye Bilateral Attention

---

In [ ]:
# ============================================================
# CELL 23 — FINAL CHECK AND SAVE DISEASE-AWARE MEMORY
# ============================================================

# ------------------------------------------------------------
# FINAL INTEGRITY CHECKS
# ------------------------------------------------------------

final_checks = {

    "Correct feature shape":
        tuple(all_disease_aware_features.shape)
        == (len(train_features), FEATURE_DIM),

    "Correct attention shape":
        tuple(all_trained_attention_weights.shape)
        == (len(train_features), NUM_CLASSES),

    "All disease-aware features finite":
        torch.isfinite(
            all_disease_aware_features
        ).all().item(),

    "All attention weights finite":
        torch.isfinite(
            all_trained_attention_weights
        ).all().item(),

    "No negative attention weights":
        torch.all(
            all_trained_attention_weights >= 0
        ).item(),

    "Attention rows sum to 1":
        torch.allclose(
            all_trained_attention_weights.sum(dim=1),
            torch.ones(
                len(all_trained_attention_weights)
            ),
            atol=1e-6
        )
}


print("=" * 70)
print("FINAL DISEASE-AWARE MEMORY CHECK")
print("=" * 70)

for check, result in final_checks.items():

    print(
        f"{'PASS' if result else 'FAIL'} — {check}"
    )

assert all(final_checks.values())


# ------------------------------------------------------------
# SAVE TRAINED FEATURE MEMORY
# ------------------------------------------------------------

DISEASE_AWARE_FEATURES_PATH = os.path.join(
    ROOT,
    "disease_aware_features.pt"
)

DISEASE_ATTENTION_WEIGHTS_PATH = os.path.join(
    ROOT,
    "disease_attention_weights.pt"
)


torch.save(
    all_disease_aware_features,
    DISEASE_AWARE_FEATURES_PATH
)

torch.save(
    all_trained_attention_weights,
    DISEASE_ATTENTION_WEIGHTS_PATH
)


assert os.path.exists(
    DISEASE_AWARE_FEATURES_PATH
)

assert os.path.exists(
    DISEASE_ATTENTION_WEIGHTS_PATH
)


print("\n" + "=" * 70)
print("DISEASE-AWARE ATTENTION MODULE COMPLETE")
print("=" * 70)

print(
    "\nSaved feature memory:",
    DISEASE_AWARE_FEATURES_PATH
)

print(
    "Saved attention weights:",
    DISEASE_ATTENTION_WEIGHTS_PATH
)

print(
    "\nFinal feature shape:",
    all_disease_aware_features.shape
)

print(
    "Final attention shape:",
    all_trained_attention_weights.shape
)

FINAL DISEASE-AWARE MEMORY CHECK
PASS — Correct feature shape
PASS — Correct attention shape
PASS — All disease-aware features finite
PASS — All attention weights finite
PASS — No negative attention weights
PASS — Attention rows sum to 1

DISEASE-AWARE ATTENTION MODULE COMPLETE

Saved feature memory: /content/drive/My Drive/Eye Disease/Dataset/disease_aware_features.pt
Saved attention weights: /content/drive/My Drive/Eye Disease/Dataset/disease_attention_weights.pt

Final feature shape: torch.Size([4491, 768])
Final attention shape: torch.Size([4491, 8])


CELL 24 — AVERAGE ATTENTION PER DISEASE PROTOTYPE


The sample inspection suggested that attention may be concentrating heavily
on prototype C.

We now calculate the average attention assigned to each disease prototype
across all 4491 samples.

This provides a global view of prototype usage.

A healthy disease-aware attention mechanism does not necessarily require
equal attention across all prototypes. However, extreme dominance by one
prototype may indicate attention collapse.

---

In [ ]:
# ============================================================
# CELL 24 — AVERAGE ATTENTION PER DISEASE PROTOTYPE
# ============================================================

average_attention = (
    all_trained_attention_weights
    .mean(dim=0)
)

print("=" * 70)
print("AVERAGE ATTENTION ACROSS ALL SAMPLES")
print("=" * 70)

for disease, attention in zip(
    LABEL_COLUMNS,
    average_attention
):
    print(
        f"{disease}: {attention.item():.6f}"
    )

print("\nTotal:", average_attention.sum().item())

assert torch.allclose(
    average_attention.sum(),
    torch.tensor(1.0),
    atol=1e-6
)

AVERAGE ATTENTION ACROSS ALL SAMPLES
N: 0.069433
D: 0.065359
G: 0.136228
C: 0.380563
A: 0.077157
H: 0.067030
M: 0.139301
O: 0.064929

Total: 1.0


 CELL 25 — PROTOTYPE DOMINANCE FREQUENCY

For every sample, we identify the disease prototype with the highest
attention weight.

We then count how frequently each prototype receives maximum attention
across all 4491 samples.

This helps determine whether different samples dynamically select
different disease prototypes or whether one prototype dominates most
of the dataset.

---

In [ ]:
# ============================================================
# CELL 25 — PROTOTYPE DOMINANCE FREQUENCY
# ============================================================

dominant_prototypes = torch.argmax(
    all_trained_attention_weights,
    dim=1
)

dominant_counts = torch.bincount(
    dominant_prototypes,
    minlength=NUM_CLASSES
)

dominant_percentages = (
    dominant_counts.float()
    / len(all_trained_attention_weights)
    * 100
)

print("=" * 70)
print("MAXIMUM-ATTENTION PROTOTYPE FREQUENCY")
print("=" * 70)

for i, disease in enumerate(LABEL_COLUMNS):

    print(
        f"{disease}: "
        f"{dominant_counts[i].item():4d} samples "
        f"({dominant_percentages[i].item():6.2f}%)"
    )

print(
    "\nTotal samples:",
    dominant_counts.sum().item()
)

MAXIMUM-ATTENTION PROTOTYPE FREQUENCY
N:    0 samples (  0.00%)
D:    0 samples (  0.00%)
G:    5 samples (  0.11%)
C: 4299 samples ( 95.72%)
A:    0 samples (  0.00%)
H:  181 samples (  4.03%)
M:    5 samples (  0.11%)
O:    1 samples (  0.02%)

Total samples: 4491


 CELL 26 — ATTENTION ENTROPY ANALYSIS


Attention entropy measures how concentrated or distributed the prototype
attention is.

For 8 disease prototypes:

    Maximum entropy:
        log(8) ≈ 2.079

This corresponds approximately to uniform attention:

    [0.125, 0.125, ..., 0.125]

Low entropy indicates concentrated attention, where one or a small number
of prototypes receive most of the attention.

We calculate entropy for every sample and summarize its distribution.

---

In [ ]:
# ============================================================
# CELL 26 — ATTENTION ENTROPY ANALYSIS
# ============================================================

epsilon = 1e-12

attention_entropy = -(
    all_trained_attention_weights
    * torch.log(
        all_trained_attention_weights + epsilon
    )
).sum(dim=1)

max_entropy = np.log(NUM_CLASSES)

print("=" * 70)
print("ATTENTION ENTROPY")
print("=" * 70)

print(
    f"Minimum entropy: "
    f"{attention_entropy.min().item():.6f}"
)

print(
    f"Maximum entropy: "
    f"{attention_entropy.max().item():.6f}"
)

print(
    f"Mean entropy: "
    f"{attention_entropy.mean().item():.6f}"
)

print(
    f"Maximum possible entropy: "
    f"{max_entropy:.6f}"
)

print(
    f"\nNormalized mean entropy: "
    f"{(attention_entropy.mean() / max_entropy).item():.4f}"
)

ATTENTION ENTROPY
Minimum entropy: 1.782084
Maximum entropy: 2.077278
Mean entropy: 1.810671
Maximum possible entropy: 2.079442

Normalized mean entropy: 0.8707


 CELL 27 — ATTENTION PATTERNS BY TRUE DISEASE LABEL


We compare the average attention distribution for samples containing each
disease label.

For every disease:

    • Select all samples where that disease is present
    • Calculate the mean attention over all 8 prototypes

This does not treat attention weights as disease probabilities.

Instead, it helps determine whether the learned Disease-Aware Attention
produces different prototype-attention patterns for different disease
groups.


In [ ]:
# ============================================================
# CELL 27 — ATTENTION PATTERNS BY TRUE DISEASE LABEL
# ============================================================

print("=" * 90)
print("AVERAGE ATTENTION FOR SAMPLES CONTAINING EACH DISEASE")
print("=" * 90)

label_attention_profiles = {}

for disease_index, disease_name in enumerate(
    LABEL_COLUMNS
):

    disease_mask = (
        train_labels[:, disease_index] == 1
    )

    disease_attention_profile = (
        all_trained_attention_weights[
            disease_mask
        ]
        .mean(dim=0)
    )

    label_attention_profiles[
        disease_name
    ] = disease_attention_profile

    dominant_disease = LABEL_COLUMNS[
        torch.argmax(
            disease_attention_profile
        ).item()
    ]

    print(
        f"\nTrue Disease Group: {disease_name}"
    )

    print(
        f"Number of samples: "
        f"{disease_mask.sum().item()}"
    )

    print(
        f"Dominant attended prototype: "
        f"{dominant_disease}"
    )

    print(
        "Average attention:"
    )

    for prototype_name, weight in zip(
        LABEL_COLUMNS,
        disease_attention_profile
    ):

        print(
            f"  {prototype_name}: "
            f"{weight.item():.4f}"
        )

AVERAGE ATTENTION FOR SAMPLES CONTAINING EACH DISEASE

True Disease Group: N
Number of samples: 1502
Dominant attended prototype: C
Average attention:
  N: 0.0655
  D: 0.0603
  G: 0.1375
  C: 0.3985
  A: 0.0745
  H: 0.0618
  M: 0.1415
  O: 0.0604

True Disease Group: D
Number of samples: 1491
Dominant attended prototype: C
Average attention:
  N: 0.0670
  D: 0.0622
  G: 0.1376
  C: 0.3921
  A: 0.0752
  H: 0.0638
  M: 0.1401
  O: 0.0621

True Disease Group: G
Number of samples: 269
Dominant attended prototype: C
Average attention:
  N: 0.0661
  D: 0.0615
  G: 0.1377
  C: 0.3870
  A: 0.0771
  H: 0.0631
  M: 0.1458
  O: 0.0617

True Disease Group: C
Number of samples: 286
Dominant attended prototype: C
Average attention:
  N: 0.1175
  D: 0.1321
  G: 0.0946
  C: 0.1837
  A: 0.1090
  H: 0.1354
  M: 0.1042
  O: 0.1234

True Disease Group: A
Number of samples: 215
Dominant attended prototype: C
Average attention:
  N: 0.0660
  D: 0.0608
  G: 0.1397
  C: 0.3927
  A: 0.0752
  H: 0.0624
  M: 0.1

CELL 28 — ATTENTION BEHAVIOR SUMMARY


We summarize the global behavior of the trained Disease-Aware Attention
mechanism.

The summary combines:

    • Most dominant prototype
    • Percentage of samples dominated by that prototype
    • Average attention assigned to that prototype
    • Mean normalized attention entropy

These measurements will guide the next decision:

    A) Attention is sufficiently diverse
       → Keep the current module

    B) Attention shows severe prototype collapse
       → Apply a small training stabilization enhancement and retrain

The proposed Disease-Aware Attention architecture itself is not changed.
Only the training objective or attention stabilization strategy would be
adjusted if required.

---

In [ ]:
# ============================================================
# CELL 28 — ATTENTION BEHAVIOR SUMMARY
# ============================================================

most_dominant_index = torch.argmax(
    dominant_counts
).item()

most_dominant_name = (
    LABEL_COLUMNS[most_dominant_index]
)

most_dominant_percentage = (
    dominant_percentages[
        most_dominant_index
    ].item()
)

most_dominant_average_attention = (
    average_attention[
        most_dominant_index
    ].item()
)

normalized_mean_entropy = (
    attention_entropy.mean()
    / max_entropy
).item()


print("=" * 70)
print("DISEASE-AWARE ATTENTION BEHAVIOR SUMMARY")
print("=" * 70)

print(
    f"Most dominant prototype: "
    f"{most_dominant_name}"
)

print(
    f"Dominant for: "
    f"{most_dominant_percentage:.2f}% "
    f"of all samples"
)

print(
    f"Average attention received: "
    f"{most_dominant_average_attention:.6f}"
)

print(
    f"Normalized mean entropy: "
    f"{normalized_mean_entropy:.4f}"
)


print("\nInterpretation:")

if most_dominant_percentage > 80:

    print(
        "WARNING — Strong prototype dominance detected."
    )

elif most_dominant_percentage > 50:

    print(
        "NOTICE — Moderate prototype dominance detected."
    )

else:

    print(
        "GOOD — Attention dominance is distributed "
        "across multiple prototypes."
    )


if normalized_mean_entropy < 0.30:

    print(
        "WARNING — Attention distributions are highly "
        "concentrated."
    )

elif normalized_mean_entropy < 0.60:

    print(
        "NOTICE — Attention distributions show moderate "
        "concentration."
    )

else:

    print(
        "GOOD — Attention distributions retain substantial "
        "diversity."
    )

DISEASE-AWARE ATTENTION BEHAVIOR SUMMARY
Most dominant prototype: C
Dominant for: 95.72% of all samples
Average attention received: 0.380563
Normalized mean entropy: 0.8707

Interpretation:
WARNING — Strong prototype dominance detected.
GOOD — Attention distributions retain substantial diversity.


CELL 29 — INSPECT LEARNED DISEASE BIAS


Disease-Aware Attention includes a learnable disease-specific bias vector.

The bias contains one value for each disease prototype:

    [N, D, G, C, A, H, M, O]

The same bias is added to the attention logits of every sample.

This inspection determines whether one disease prototype has developed
a disproportionately large global bias during training.

A large positive bias for Cataract (C), combined with the softmax
operation, could contribute to global prototype dominance.

---

In [ ]:
# # ============================================================
# # CELL 29 — INSPECT LEARNED DISEASE BIAS
# # ============================================================

# learned_disease_bias = (
#     best_daa_model
#     .attention
#     .disease_bias
#     .detach()
#     .cpu()
# )

# print("=" * 70)
# print("LEARNED DISEASE BIAS")
# print("=" * 70)

# for disease, bias_value in zip(
#     LABEL_COLUMNS,
#     learned_disease_bias
# ):

#     print(
#         f"{disease}: "
#         f"{bias_value.item():.6f}"
#     )


# print("\n" + "=" * 70)
# print("BIAS SUMMARY")
# print("=" * 70)

# print(
#     f"Minimum bias: "
#     f"{learned_disease_bias.min().item():.6f}"
# )

# print(
#     f"Maximum bias: "
#     f"{learned_disease_bias.max().item():.6f}"
# )

# print(
#     f"Mean bias: "
#     f"{learned_disease_bias.mean().item():.6f}"
# )

# print(
#     f"Bias standard deviation: "
#     f"{learned_disease_bias.std().item():.6f}"
# )


# max_bias_index = torch.argmax(
#     learned_disease_bias
# ).item()

# min_bias_index = torch.argmin(
#     learned_disease_bias
# ).item()


# print(
#     "\nHighest bias prototype:",
#     LABEL_COLUMNS[max_bias_index]
# )

# print(
#     "Lowest bias prototype:",
#     LABEL_COLUMNS[min_bias_index]
# )


CELL 30 — RELATIVE DISEASE BIAS ANALYSIS

Attention is influenced by relative logit differences.

We therefore compare every disease bias against the mean disease bias.

A positive relative bias indicates a prototype receives a global
advantage relative to the average.

A negative relative bias indicates a relative disadvantage.

If Cataract (C) has a substantially larger relative bias than the other
prototypes, the global disease bias may be contributing directly to the
observed attention collapse.


In [ ]:
# # ============================================================
# # CELL 30 — RELATIVE DISEASE BIAS ANALYSIS
# # ============================================================

# mean_bias = learned_disease_bias.mean()

# relative_disease_bias = (
#     learned_disease_bias
#     - mean_bias
# )

# print("=" * 70)
# print("RELATIVE DISEASE BIAS")
# print("=" * 70)

# for disease, relative_bias in zip(
#     LABEL_COLUMNS,
#     relative_disease_bias
# ):

#     print(
#         f"{disease}: "
#         f"{relative_bias.item():+.6f}"
#     )


# bias_range = (
#     learned_disease_bias.max()
#     - learned_disease_bias.min()
# )

# print("\nBias range:", bias_range.item())


# c_index = LABEL_COLUMNS.index("C")

# c_bias = learned_disease_bias[c_index]

# other_biases = torch.cat([
#     learned_disease_bias[:c_index],
#     learned_disease_bias[c_index + 1:]
# ])

# c_advantage_over_others = (
#     c_bias
#     - other_biases.mean()
# )

# print(
#     "\nCataract bias:",
#     c_bias.item()
# )

# print(
#     "Average bias of other prototypes:",
#     other_biases.mean().item()
# )

# print(
#     "Cataract bias advantage:",
#     c_advantage_over_others.item()
# )


CELL 31 — BIAS AND ATTENTION DOMINANCE COMPARISON

We compare the learned disease bias with the average attention received
by each prototype.

This does not prove causation by itself.

However, if one prototype has both:

    • the largest positive bias
    • overwhelmingly dominant average attention

then the global disease bias is a plausible contributor to attention
collapse.

This comparison helps determine whether the next stabilization step should
focus on the disease bias or on the attention-logit scaling.

---

In [ ]:
# # ============================================================
# # CELL 31 — BIAS AND ATTENTION DOMINANCE COMPARISON
# # ============================================================

# print("=" * 85)
# print("DISEASE BIAS VS AVERAGE ATTENTION")
# print("=" * 85)

# print(
#     f"{'Disease':<10}"
#     f"{'Bias':>15}"
#     f"{'Avg Attention':>20}"
# )

# print("-" * 85)

# for i, disease in enumerate(
#     LABEL_COLUMNS
# ):

#     print(
#         f"{disease:<10}"
#         f"{learned_disease_bias[i].item():>15.6f}"
#         f"{average_attention[i].item():>20.6f}"
#     )


# bias_max_disease = LABEL_COLUMNS[
#     torch.argmax(
#         learned_disease_bias
#     ).item()
# ]

# attention_max_disease = LABEL_COLUMNS[
#     torch.argmax(
#         average_attention
#     ).item()
# ]


# print("\n" + "=" * 85)

# print(
#     "Prototype with highest bias:",
#     bias_max_disease
# )

# print(
#     "Prototype with highest average attention:",
#     attention_max_disease
# )


# if bias_max_disease == attention_max_disease:

#     print(
#         "\nNOTICE — The same prototype has both the highest "
#         "bias and highest average attention."
#     )

# else:

#     print(
#         "\nThe most dominant attention prototype does not "
#         "have the highest disease bias."
#     )

CELL 32 — RAW ATTENTION LOGIT INSPECTION

This diagnostic inspects the attention scores before the Softmax
operation.

The Disease-Aware Attention mechanism computes:

    Q = QueryProjection(ImageFeatures)

    K = KeyProjection(DiseasePrototypes)

    RawScores = Q × Kᵀ / sqrt(AttentionDimension)

    AttentionLogits =
        RawScores + DiseaseBias

    AttentionWeights =
        Softmax(AttentionLogits)

The purpose of this diagnostic is to determine whether prototype
dominance already exists before Softmax or whether Softmax is strongly
amplifying relatively small score differences.

The inspection compares:

    • Raw query-key similarity scores
    • Scores after disease bias
    • Score ranges across prototypes
    • Cataract score relative to other prototypes

This diagnostic does not modify the model.

---

In [ ]:
# ============================================================
# CELL 32 — NORMALIZED ATTENTION SCORE INSPECTION
# ============================================================

best_daa_model.eval()

attention_module = best_daa_model.attention


with torch.no_grad():

    # --------------------------------------------------------
    # SELECT REPRESENTATIVE SAMPLES
    # --------------------------------------------------------

    inspection_features = (
        train_features[:512]
        .to(device)
    )


    # --------------------------------------------------------
    # COMPUTE PROJECTED QUERIES
    # --------------------------------------------------------

    queries = (
        attention_module
        .query_projection(
            inspection_features
        )
    )


    # --------------------------------------------------------
    # COMPUTE PROJECTED KEYS
    # --------------------------------------------------------

    keys = (
        attention_module
        .key_projection(
            disease_prototypes.to(device)
        )
    )


    # --------------------------------------------------------
    # NORMALIZE QUERIES AND KEYS
    #
    # IMPORTANT:
    # This exactly matches the updated
    # DiseaseAwareAttention forward pass.
    # --------------------------------------------------------

    normalized_queries = F.normalize(
        queries,
        p=2,
        dim=1
    )

    normalized_keys = F.normalize(
        keys,
        p=2,
        dim=1
    )


    # --------------------------------------------------------
    # NORMALIZED QUERY-KEY SIMILARITY
    #
    # This is cosine similarity.
    #
    # Expected approximate range:
    # [-1, 1]
    # --------------------------------------------------------

    raw_attention_scores = (
        normalized_queries
        @ normalized_keys.T
    )


    # --------------------------------------------------------
    # ADD LEARNED DISEASE BIAS
    #
    # Removed as per updated DiseaseAwareAttention class.
    # Logits are now just the raw scores.
    # --------------------------------------------------------

    attention_logits = raw_attention_scores


    # --------------------------------------------------------
    # RECONSTRUCT ATTENTION
    #
    # IMPORTANT:
    # Do NOT apply dropout here because
    # the model is in evaluation mode and
    # we want the actual deterministic
    # attention distribution.
    # --------------------------------------------------------

    reconstructed_attention = F.softmax(
        attention_logits,
        dim=1
    )


# ============================================================
# OUTPUT
# ============================================================

print("=" * 75)
print("NORMALIZED ATTENTION SCORE INSPECTION")
print("=" * 75)

print(
    "Inspection samples:",
    inspection_features.size(0)
)

print(
    "Number of prototypes:",
    len(LABEL_COLUMNS)
)


# ------------------------------------------------------------
# AVERAGE SCORES
# ------------------------------------------------------------

average_raw_scores = (
    raw_attention_scores
    .mean(dim=0)
    .cpu()
)

average_attention_logits = (
    attention_logits
    .mean(dim=0)
    .cpu()
)

average_reconstructed_attention = (
    reconstructed_attention
    .mean(dim=0)
    .cpu()
)


print("\nAVERAGE NORMALIZED QUERY-KEY SCORES")

for disease, value in zip(
    LABEL_COLUMNS,
    average_raw_scores
):

    print(
        f"{disease}: "
        f"{value.item():.6f}"
    )


print("\nAVERAGE ATTENTION LOGITS AFTER BIAS")

for disease, value in zip(
    LABEL_COLUMNS,
    average_attention_logits
):

    print(
        f"{disease}: "
        f"{value.item():.6f}"
    )


print("\nAVERAGE RECONSTRUCTED ATTENTION")

for disease, value in zip(
    LABEL_COLUMNS,
    average_reconstructed_attention
):

    print(
        f"{disease}: "
        f"{value.item():.6f}"
    )


# ------------------------------------------------------------
# SCORE MAGNITUDE ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("NORMALIZED SCORE MAGNITUDE ANALYSIS")
print("=" * 75)

print(
    f"Raw normalized score minimum: "
    f"{raw_attention_scores.min().item():.6f}"
)

print(
    f"Raw normalized score maximum: "
    f"{raw_attention_scores.max().item():.6f}"
)

print(
    f"Raw normalized score standard deviation: "
    f"{raw_attention_scores.std().item():.6f}"
)


print(
    f"\nLogit minimum: "
    f"{attention_logits.min().item():.6f}"
)

print(
    f"Logit maximum: "
    f"{attention_logits.max().item():.6f}"
)

print(
    f"Logit standard deviation: "
    f"{attention_logits.std().item():.6f}"
)


# ------------------------------------------------------------
# VERIFY NORMALIZATION
# ------------------------------------------------------------

query_norms = torch.norm(
    normalized_queries,
    p=2,
    dim=1
)

key_norms = torch.norm(
    normalized_keys,
    p=2,
    dim=1
)


print("\n" + "=" * 75)
print("NORMALIZATION INTEGRITY CHECK")
print("=" * 75)

print(
    f"Query norm minimum: "
    f"{query_norms.min().item():.6f}"
)

print(
    f"Query norm maximum: "
    f"{query_norms.max().item():.6f}"
)

print(
    f"Query norm mean: "
    f"{query_norms.mean().item():.6f}"
)

print()

print(
    f"Key norm minimum: "
    f"{key_norms.min().item():.6f}"
)

print(
    f"Key norm maximum: "
    f"{key_norms.max().item():.6f}"
)

print(
    f"Key norm mean: "
    f"{key_norms.mean().item():.6f}"
)


# ------------------------------------------------------------
# CATARACT ADVANTAGE
# ------------------------------------------------------------

c_index = LABEL_COLUMNS.index("C")

c_logits = attention_logits[:, c_index]

other_logits = torch.cat(
    [
        attention_logits[:, :c_index],
        attention_logits[:, c_index + 1:]
    ],
    dim=1
)

c_advantage = (
    c_logits
    - other_logits.mean(dim=1)
)


print("\n" + "=" * 75)
print("CATARACT LOGIT ADVANTAGE")
print("=" * 75)

print(
    f"Average C logit: "
    f"{c_logits.mean().item():.6f}"
)

print(
    f"Average other logit: "
    f"{other_logits.mean().item():.6f}"
)

print(
    f"Average C advantage: "
    f"{c_advantage.mean().item():.6f}"
)

print(
    f"Maximum C advantage: "
    f"{c_advantage.max().item():.6f}"
)

print(
    f"Minimum C advantage: "
    f"{c_advantage.min().item():.6f}"
)

NORMALIZED ATTENTION SCORE INSPECTION
Inspection samples: 512
Number of prototypes: 8

AVERAGE NORMALIZED QUERY-KEY SCORES
N: -0.864942
D: -0.949581
G: -0.126167
C: 0.915719
A: -0.739873
H: -0.924045
M: -0.104788
O: -0.947672

AVERAGE ATTENTION LOGITS AFTER BIAS
N: -0.864942
D: -0.949581
G: -0.126167
C: 0.915719
A: -0.739873
H: -0.924045
M: -0.104788
O: -0.947672

AVERAGE RECONSTRUCTED ATTENTION
N: 0.066177
D: 0.061076
G: 0.138355
C: 0.394704
A: 0.074818
H: 0.062670
M: 0.141129
O: 0.061071

NORMALIZED SCORE MAGNITUDE ANALYSIS
Raw normalized score minimum: -0.983944
Raw normalized score maximum: 0.960545
Raw normalized score standard deviation: 0.630456

Logit minimum: -0.983944
Logit maximum: 0.960545
Logit standard deviation: 0.630456

NORMALIZATION INTEGRITY CHECK
Query norm minimum: 1.000000
Query norm maximum: 1.000000
Query norm mean: 1.000000

Key norm minimum: 1.000000
Key norm maximum: 1.000000
Key norm mean: 1.000000

CATARACT LOGIT ADVANTAGE
Average C logit: 0.915719
Average 

 CELL 33 — ATTENTION COLLAPSE SOURCE SUMMARY

This diagnostic compares prototype dominance at three stages:

    Stage 1:
        Raw query-key similarity scores

    Stage 2:
        Attention logits after disease bias

    Stage 3:
        Final attention after Softmax

If Cataract already dominates Stage 1, the primary cause is the learned
query-key similarity.

If Cataract becomes dominant mainly after adding the bias, the disease
bias contributes substantially.

If the raw and biased scores are relatively close but the final
attention becomes extremely concentrated, Softmax amplification and
attention-logit scale are likely contributing to the collapse.

---

In [ ]:
# # ============================================================
# # CELL 33 — ATTENTION DOMINANCE COMPARISON
# # ============================================================

# # ------------------------------------------------------------
# # 1. RAW NORMALIZED QUERY-KEY SCORES
# # ------------------------------------------------------------

# raw_max_indices = torch.argmax(
#     raw_attention_scores,
#     dim=1
# )


# # ------------------------------------------------------------
# # 2. SCORES AFTER DISEASE BIAS
# # ------------------------------------------------------------

# logit_max_indices = torch.argmax(
#     attention_logits,
#     dim=1
# )


# # ------------------------------------------------------------
# # 3. ATTENTION WITH DISEASE BIAS
# # ------------------------------------------------------------

# attention_with_bias = F.softmax(
#     attention_logits,
#     dim=1
# )

# attention_with_bias_max_indices = torch.argmax(
#     attention_with_bias,
#     dim=1
# )


# # ------------------------------------------------------------
# # 4. ATTENTION WITHOUT DISEASE BIAS
# #
# # Directly apply Softmax to normalized Q-K scores.
# #
# # This lets us inspect the same trained model
# # without changing or retraining anything.
# # ------------------------------------------------------------

# attention_without_bias = F.softmax(
#     raw_attention_scores,
#     dim=1
# )

# attention_without_bias_max_indices = torch.argmax(
#     attention_without_bias,
#     dim=1
# )


# # ------------------------------------------------------------
# # COUNT MAXIMUM-ATTENTION PROTOTYPES
# # ------------------------------------------------------------

# raw_max_counts = torch.bincount(
#     raw_max_indices,
#     minlength=len(LABEL_COLUMNS)
# )

# logit_max_counts = torch.bincount(
#     logit_max_indices,
#     minlength=len(LABEL_COLUMNS)
# )

# with_bias_counts = torch.bincount(
#     attention_with_bias_max_indices,
#     minlength=len(LABEL_COLUMNS)
# )

# without_bias_counts = torch.bincount(
#     attention_without_bias_max_indices,
#     minlength=len(LABEL_COLUMNS)
# )


# num_samples = inspection_features.size(0)


# # ============================================================
# # DOMINANCE SUMMARY
# # ============================================================

# print("=" * 105)
# print("ATTENTION DOMINANCE COMPARISON")
# print("=" * 105)

# print(
#     f"{'Disease':<10}"
#     f"{'Raw Score':>15}"
#     f"{'+ Bias':>15}"
#     f"{'Softmax + Bias':>20}"
#     f"{'Softmax No Bias':>20}"
# )

# print("-" * 105)


# for i, disease in enumerate(LABEL_COLUMNS):

#     raw_percentage = (
#         raw_max_counts[i].item()
#         / num_samples
#         * 100
#     )

#     logit_percentage = (
#         logit_max_counts[i].item()
#         / num_samples
#         * 100
#     )

#     with_bias_percentage = (
#         with_bias_counts[i].item()
#         / num_samples
#         * 100
#     )

#     without_bias_percentage = (
#         without_bias_counts[i].item()
#         / num_samples
#         * 100
#     )

#     print(
#         f"{disease:<10}"
#         f"{raw_percentage:>14.2f}%"
#         f"{logit_percentage:>14.2f}%"
#         f"{with_bias_percentage:>19.2f}%"
#         f"{without_bias_percentage:>19.2f}%"
#     )


# # ============================================================
# # CATARACT DOMINANCE
# # ============================================================

# print("\n" + "=" * 105)
# print("CATARACT DOMINANCE COMPARISON")
# print("=" * 105)

# print(
#     f"Raw normalized similarity: "
#     f"{raw_max_counts[c_index].item() / num_samples * 100:.2f}%"
# )

# print(
#     f"After disease bias: "
#     f"{logit_max_counts[c_index].item() / num_samples * 100:.2f}%"
# )

# print(
#     f"Softmax WITH bias: "
#     f"{with_bias_counts[c_index].item() / num_samples * 100:.2f}%"
# )

# print(
#     f"Softmax WITHOUT bias: "
#     f"{without_bias_counts[c_index].item() / num_samples * 100:.2f}%"
# )


# # ============================================================
# # AVERAGE ATTENTION COMPARISON
# # ============================================================

# average_attention_with_bias = (
#     attention_with_bias
#     .mean(dim=0)
#     .cpu()
# )

# average_attention_without_bias = (
#     attention_without_bias
#     .mean(dim=0)
#     .cpu()
# )


# print("\n" + "=" * 105)
# print("AVERAGE ATTENTION DISTRIBUTION")
# print("=" * 105)

# print(
#     f"{'Disease':<10}"
#     f"{'With Bias':>20}"
#     f"{'Without Bias':>20}"
# )

# print("-" * 55)


# for i, disease in enumerate(LABEL_COLUMNS):

#     print(
#         f"{disease:<10}"
#         f"{average_attention_with_bias[i].item():>20.6f}"
#         f"{average_attention_without_bias[i].item():>20.6f}"
#     )


# # ============================================================
# # ATTENTION ENTROPY COMPARISON
# # ============================================================

# EPSILON = 1e-8


# entropy_with_bias = -torch.sum(
#     attention_with_bias
#     * torch.log(
#         attention_with_bias + EPSILON
#     ),
#     dim=1
# )


# entropy_without_bias = -torch.sum(
#     attention_without_bias
#     * torch.log(
#         attention_without_bias + EPSILON
#     ),
#     dim=1
# )


# max_entropy = torch.log(
#     torch.tensor(
#         len(LABEL_COLUMNS),
#         dtype=torch.float32,
#         device=device
#     )
# )


# print("\n" + "=" * 105)
# print("ATTENTION ENTROPY COMPARISON")
# print("=" * 105)

# print(
#     f"Mean entropy WITH bias: "
#     f"{entropy_with_bias.mean().item():.6f}"
# )

# print(
#     f"Normalized mean entropy WITH bias: "
#     f"{(entropy_with_bias.mean() / max_entropy).item():.4f}"
# )

# print()

# print(
#     f"Mean entropy WITHOUT bias: "
#     f"{entropy_without_bias.mean().item():.6f}"
# )

# print(
#     f"Normalized mean entropy WITHOUT bias: "
#     f"{(entropy_without_bias.mean() / max_entropy).item():.4f}"
# )

ATTENTION DOMINANCE COMPARISON
Disease         Raw Score         + Bias      Softmax + Bias     Softmax No Bias
---------------------------------------------------------------------------------------------------------
N                   0.00%          0.00%               0.00%               0.00%
D                   0.00%          0.00%               0.00%               0.00%
G                   0.00%          0.00%               0.00%               0.00%
C                  99.02%         99.02%              99.02%              99.02%
A                   0.00%          0.00%               0.00%               0.00%
H                   0.98%          0.98%               0.98%               0.98%
M                   0.00%          0.00%               0.00%               0.00%
O                   0.00%          0.00%               0.00%               0.00%

CATARACT DOMINANCE COMPARISON
Raw normalized similarity: 99.02%
After disease bias: 99.02%
Softmax WITH bias: 99.02%
Softmax WITHOUT 

 Cell 34 — Query–Key Representation Diagnostic

The previous attention analysis showed that Cataract (`C`) is selected as the
maximum-similarity disease prototype for the majority of inspected samples,
even after L2 normalization and even when the disease-specific bias is removed.

To understand the source of this behavior, this diagnostic examines the learned
geometry of the query and key representations.

Specifically, we investigate:

1. **Prototype Key Similarity**
   - Measures cosine similarity between every pair of learned disease keys.
   - Helps determine whether the Cataract key is unusually aligned or separated
     from the other disease prototype keys.

2. **Average Query-to-Key Similarity**
   - Measures how the image queries align, on average, with each disease key.
   - Identifies whether Cataract receives systematically higher similarity scores
     across the inspected image samples.

3. **True Disease Group Similarity**
   - Groups samples according to their ground-truth disease labels and measures
     their average similarity to each disease prototype.
   - Helps determine whether disease-specific query representations show meaningful
     alignment with their corresponding disease prototypes.

This analysis does not modify the architecture, model parameters, prototype
memory, or training process. It is a diagnostic-only investigation intended to
identify the source of prototype dominance before making any further changes.

---


In [ ]:
# ============================================================
# CELL 34 — QUERY–KEY REPRESENTATION DIAGNOSTIC
# ============================================================

best_daa_model.eval()

attention_module = best_daa_model.attention


with torch.no_grad():

    # --------------------------------------------------------
    # INSPECTION FEATURES
    # --------------------------------------------------------

    inspection_features = (
        train_features[:512]
        .to(device)
    )


    # --------------------------------------------------------
    # PROJECT IMAGE FEATURES → QUERIES
    # --------------------------------------------------------

    queries = (
        attention_module
        .query_projection(
            inspection_features
        )
    )


    # --------------------------------------------------------
    # PROJECT DISEASE PROTOTYPES → KEYS
    # --------------------------------------------------------

    keys = (
        attention_module
        .key_projection(
            disease_prototypes.to(device)
        )
    )


    # --------------------------------------------------------
    # L2 NORMALIZATION
    #
    # This exactly matches the active attention mechanism.
    # --------------------------------------------------------

    normalized_queries = F.normalize(
        queries,
        p=2,
        dim=1
    )

    normalized_keys = F.normalize(
        keys,
        p=2,
        dim=1
    )


    # --------------------------------------------------------
    # QUERY → KEY COSINE SIMILARITY
    #
    # Shape:
    # [512, 8]
    # --------------------------------------------------------

    query_key_similarity = (
        normalized_queries
        @ normalized_keys.T
    )


    # --------------------------------------------------------
    # KEY → KEY COSINE SIMILARITY
    #
    # Shape:
    # [8, 8]
    # --------------------------------------------------------

    key_similarity_matrix = (
        normalized_keys
        @ normalized_keys.T
    )


# ============================================================
# 1. PROTOTYPE KEY SIMILARITY
# ============================================================

print("=" * 80)
print("LEARNED DISEASE KEY SIMILARITY")
print("=" * 80)

print(
    f"{'':<8}",
    end=""
)

for disease in LABEL_COLUMNS:

    print(
        f"{disease:>9}",
        end=""
    )

print()


for i, disease in enumerate(LABEL_COLUMNS):

    print(
        f"{disease:<8}",
        end=""
    )

    for j in range(len(LABEL_COLUMNS)):

        print(
            f"{key_similarity_matrix[i, j].item():>9.3f}",
            end=""
        )

    print()


# ============================================================
# 2. AVERAGE QUERY → KEY SIMILARITY
# ============================================================

average_query_key_similarity = (
    query_key_similarity
    .mean(dim=0)
    .cpu()
)


print("\n" + "=" * 80)
print("AVERAGE QUERY-TO-KEY COSINE SIMILARITY")
print("=" * 80)


for disease, similarity in zip(
    LABEL_COLUMNS,
    average_query_key_similarity
):

    print(
        f"{disease}: "
        f"{similarity.item():.6f}"
    )


highest_similarity_index = torch.argmax(
    average_query_key_similarity
).item()


print()

print(
    "Highest average similarity prototype:",
    LABEL_COLUMNS[
        highest_similarity_index
    ]
)


# ============================================================
# 3. TRUE DISEASE GROUP SIMILARITY
# ============================================================

inspection_labels = (
    train_labels[:512]
    .to(device)
)


print("\n" + "=" * 80)
print("TRUE DISEASE GROUP → AVERAGE QUERY-KEY SIMILARITY")
print("=" * 80)


for disease_index, disease_name in enumerate(
    LABEL_COLUMNS
):

    disease_mask = (
        inspection_labels[
            :,
            disease_index
        ] == 1
    )


    num_disease_samples = (
        disease_mask.sum().item()
    )


    if num_disease_samples == 0:

        print(
            f"\n{disease_name}: "
            "No samples in inspection subset."
        )

        continue


    disease_similarities = (
        query_key_similarity[
            disease_mask
        ]
        .mean(dim=0)
        .cpu()
    )


    dominant_index = torch.argmax(
        disease_similarities
    ).item()


    print(
        f"\nTrue Disease: "
        f"{disease_name}"
    )

    print(
        f"Samples: "
        f"{num_disease_samples}"
    )

    print(
        f"Highest similarity prototype: "
        f"{LABEL_COLUMNS[dominant_index]}"
    )


    for prototype_name, similarity in zip(
        LABEL_COLUMNS,
        disease_similarities
    ):

        print(
            f"  {prototype_name}: "
            f"{similarity.item():.4f}"
        )


# ============================================================
# 4. CATARACT KEY POSITION ANALYSIS
# ============================================================

c_index = LABEL_COLUMNS.index("C")


c_key_similarities = (
    key_similarity_matrix[
        c_index
    ]
    .clone()
)

# Exclude self-similarity.

c_key_similarities[
    c_index
] = float("nan")


average_c_key_similarity = torch.nanmean(
    c_key_similarities
)


other_key_average_similarities = []


for i in range(len(LABEL_COLUMNS)):

    similarities = (
        key_similarity_matrix[
            i
        ]
        .clone()
    )

    similarities[i] = float("nan")

    other_key_average_similarities.append(
        torch.nanmean(similarities).item()
    )


print("\n" + "=" * 80)
print("CATARACT KEY POSITION ANALYSIS")
print("=" * 80)

print(
    f"Average similarity of C key "
    f"to other keys: "
    f"{average_c_key_similarity.item():.6f}"
)

print()

for disease, similarity in zip(
    LABEL_COLUMNS,
    other_key_average_similarities
):

    print(
        f"{disease}: "
        f"{similarity:.6f}"
    )


print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

print(
    "This analysis does not modify the model "
    "or training process."
)


LEARNED DISEASE KEY SIMILARITY
                N        D        G        C        A        H        M        O
N           1.000    0.903    0.337   -0.795    0.551    0.841   -0.146    0.926
D           0.903    1.000    0.043   -0.878    0.690    0.975   -0.014    0.978
G           0.337    0.043    1.000   -0.206    0.131    0.028    0.197    0.178
C          -0.795   -0.878   -0.206    1.000   -0.756   -0.877   -0.206   -0.880
A           0.551    0.690    0.131   -0.756    1.000    0.602    0.687    0.780
H           0.841    0.975    0.028   -0.877    0.602    1.000   -0.074    0.923
M          -0.146   -0.014    0.197   -0.206    0.687   -0.074    1.000    0.122
O           0.926    0.978    0.178   -0.880    0.780    0.923    0.122    1.000

AVERAGE QUERY-TO-KEY COSINE SIMILARITY
N: -0.864942
D: -0.949581
G: -0.126167
C: 0.915719
A: -0.739873
H: -0.924045
M: -0.104788
O: -0.947672

Highest average similarity prototype: C

TRUE DISEASE GROUP → AVERAGE QUERY-KEY SIMILARITY

True